# 💘 CUPIDY — Preparación de Datos
## Proyecto CRISP-DM | Speed Dating Experiment — Columbia University

**Objetivo:** Predecir si dos personas tendrán match mutuo,
usando únicamente perfil del participante

**Dataset:** Speed Dating Experiment — 8.378 observaciones, 195 variables originales  
**Variable objetivo:** `match` (binaria: 1=match mutuo, 0=no match)   
**Metodología:** CRISP-DM (Cross Industry Standard Process for Data Mining)

In [92]:
import matplotlib
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
PALETTE = {"primario": "#FF1493", "secundario": "#FFB6C1", "negativo": "#FF6B6B"}
print("Librerías cargadas")

Librerías cargadas


## FASE 1 — ENTENDIMIENTO DEL NEGOCIO

### 1.1 Contexto
El experimento de speed dating de Columbia University (2002–2004) reunió a estudiantes
de posgrado en 21 oleadas de eventos. Cada participante tuvo entre 10 y 20 mini-citas
de 4 minutos. Al final de cada cita, marcaba YES o NO. Si ambos marcaban YES → match.

### 1.2 Problema de negocio
Apps de citas que necesitan predecir compatibilidad
**antes** de que las personas se conozcan, usando solo el perfil.

### 1.3 Objetivo de minería
Construir un modelo de clasificación binaria que prediga `match` (1/0) usando
variables del perfil del participante recopiladas en el registro.

### 1.4 Diseño de solución
| Elemento | Detalle |
|---|---|
| Tipo de problema | Clasificación binaria supervisada |
| Variable objetivo | match (0=No, 1=Sí) |
| Métodos | DT, MLP, SVM, KNN, RF, XGBoost, GradientBoosting |
| Evaluación principal | ROC-AUC (métrica correcta para clases desbalanceadas) |
| Línea base | ROC-AUC = 0.5 (clasificador aleatorio) |
| Meta | ROC-AUC > 0.75 en test set |

## FASE 2 — ENTENDIMIENTO DE LOS DATOS

### 2.1 Carga y exploración inicial

In [93]:
df = pd.read_csv("../data/Speed Dating Data.csv", encoding="latin-1")
print(f"Shape original: {df.shape}")
print(f"Columnas: {df.shape[1]}")
print(f"Filas: {df.shape[0]:,}")
print(f"\nPrimeras columnas: {df.columns[:10].tolist()}")
print(f"\nTipos de datos:")
print(df.dtypes.value_counts())
df.head(3)

Shape original: (8378, 82)
Columnas: 82
Filas: 8,378

Primeras columnas: ['iid', 'id', 'idg', 'partner', 'pid', 'wave', 'round', 'position', 'positin1', 'order']

Tipos de datos:
int64      77
object      4
float64     1
Name: count, dtype: int64


,iid,id,idg,partner,pid,wave,round,position,positin1,order,condtn,undergrd,zipcode,career,from,field,gender,age,age_o,race,race_o,samerace,imprace,goal,date,go_out,field_cd,career_c,income,mn_sat,tuition,exphappy,expnum,attr1_1,sinc1_1,intel1_1,fun1_1,amb1_1,shar1_1,attr3_1,sinc3_1,intel3_1,fun3_1,amb3_1,attr,sinc,intel,fun,amb,shar,like,prob,met,attr_o,sinc_o,intel_o,fun_o,amb_o,shar_o,like_o,prob_o,dec,dec_o,int_corr,sports,tvsports,exercise,dining,museums,art,hiking,gaming,clubbing,reading,tv,theater,movies,concerts,music,shopping,yoga,match
0,1,3,224,249,66,2,11,7,5,3,2,NYU,12611,Law,FL,Law,0,30,35,6,4,0,6,5,1,6,5,2,47584,1489,12043,8,1,55,92,68,58,13,40,1,2,7,10,9,3,7,6,5,1,5,5,5,2,3,2,10,5,6,4,2,2,0,0,-0.393814,1,2,5,10,1,2,5,8,3,2,3,6,2,9,9,3,7,0
1,2,191,46,166,146,19,10,12,19,10,2,NYU,15038,Medicine,TX,Law,1,37,33,1,1,1,10,2,4,4,5,16,119062,1079,36895,8,5,8,63,4,66,32,80,4,7,2,5,4,1,2,1,4,3,3,4,4,1,3,5,2,5,1,1,5,1,0,0,0.888965,7,7,8,3,1,2,7,3,4,9,7,6,9,7,2,9,10,0
2,3,401,224,215,16,21,2,6,19,9,1,NYU,12591,Arts,CA,Science,0,26,44,1,4,0,6,3,5,4,10,1,67624,827,47393,6,2,61,32,21,11,72,23,10,1,3,1,9,2,2,10,3,7,4,6,2,1,5,3,7,5,2,7,3,3,0,0,0.666983,3,3,6,9,9,3,7,9,9,6,6,8,10,8,7,7,8,0


### 2.2 Ciclo de vida de los datos
- **Generación:** Formularios físicos en eventos de speed dating (2002–2004), Columbia University
- **Momentos de captura:**
  - **Time 1 (signup):** Antes del evento — perfil, preferencias, intereses ← USAREMOS ESTAS
  - **Scorecard:** Durante cada cita — ratings mutuos
  - **Time 2/3 (followup):** Días después 
- **Almacenamiento:** CSV con 195 columnas, digitalizado desde formularios físicos
- **Periodicidad:** Histórico estático (21 waves, no se actualiza)

### 2.3 Diccionario de variables seleccionadas

In [94]:
diccionario = pd.DataFrame([
    ("gender","Género del participante","Binaria","0=Mujer, 1=Hombre","Time 1"),
    ("age","Edad del participante","Numérica","18–45","Time 1"),
    ("age_o","Edad de la pareja","Numérica","18–45","Time 1"),
    ("race","Raza del participante","Categórica","1=Negro,2=Caucásico,3=Latino,4=Asiático,6=Otro","Time 1"),
    ("race_o","Raza de la pareja","Categórica","1-6","Time 1"),
    ("samerace","Misma raza que la pareja","Binaria","0=No, 1=Sí","Time 1"),
    ("imprace","Importancia de misma raza en pareja ideal","Ordinal","1–10","Time 1"),
    ("imprelig","Importancia de misma religión en pareja","Ordinal","1–10","Time 1"),
    ("goal","Objetivo en el evento","Categórica","1=Diversión…6=Otro","Time 1"),
    ("date","Frecuencia de citas en general","Ordinal","1=Varias/sem…7=Nunca","Time 1"),
    ("go_out","Frecuencia de salidas sociales","Ordinal","1=Varias/sem…7=Nunca","Time 1"),
    ("field_cd","Área de estudio codificada","Categórica","1=Derecho…18=Otro","Time 1"),
    ("career_c","Carrera esperada codificada","Categórica","1=Abogado…17=Arquitecto","Time 1"),
    ("income","Ingreso mediano del código postal","Numérica",">0","Time 1"),
    ("mn_sat","SAT mediano de su universidad (proxy IQ)","Numérica",">0","Time 1"),
    ("tuition","Matrícula de su universidad","Numérica",">0","Time 1"),
    ("exphappy","Felicidad esperada en el evento","Ordinal","1–10","Time 1"),
    ("expnum","Personas que espera que le gusten","Numérica","≥0","Time 1"),
    ("attr1_1","Importancia atractivo en pareja ideal","Numérica","0–100 o 0–10","Time 1"),
    ("sinc1_1","Importancia sinceridad en pareja ideal","Numérica","0–100 o 0–10","Time 1"),
    ("intel1_1","Importancia inteligencia en pareja ideal","Numérica","0–100 o 0–10","Time 1"),
    ("fun1_1","Importancia diversión en pareja ideal","Numérica","0–100 o 0–10","Time 1"),
    ("amb1_1","Importancia ambición en pareja ideal","Numérica","0–100 o 0–10","Time 1"),
    ("shar1_1","Importancia intereses compartidos","Numérica","0–100 o 0–10","Time 1"),
    ("attr3_1","Autopercepción de atractivo propio","Ordinal","1–10","Time 1"),
    ("sinc3_1","Autopercepción de sinceridad propia","Ordinal","1–10","Time 1"),
    ("intel3_1","Autopercepción de inteligencia propia","Ordinal","1–10","Time 1"),
    ("fun3_1","Autopercepción de diversión propia","Ordinal","1–10","Time 1"),
    ("amb3_1","Autopercepción de ambición propia","Ordinal","1–10","Time 1"),
    ("sports","Interés en practicar deportes","Ordinal","1–10","Time 1"),
    ("tvsports","Interés en ver deportes","Ordinal","1–10","Time 1"),
    ("exercise","Interés en ejercicio/gym","Ordinal","1–10","Time 1"),
    ("dining","Interés en salir a comer","Ordinal","1–10","Time 1"),
    ("museums","Interés en museos y galerías","Ordinal","1–10","Time 1"),
    ("art","Interés en arte","Ordinal","1–10","Time 1"),
    ("hiking","Interés en senderismo/camping","Ordinal","1–10","Time 1"),
    ("gaming","Interés en videojuegos","Ordinal","1–10","Time 1"),
    ("clubbing","Interés en bailar/clubes nocturnos","Ordinal","1–10","Time 1"),
    ("reading","Interés en lectura","Ordinal","1–10","Time 1"),
    ("tv","Interés en ver televisión","Ordinal","1–10","Time 1"),
    ("theater","Interés en teatro","Ordinal","1–10","Time 1"),
    ("movies","Interés en cine","Ordinal","1–10","Time 1"),
    ("concerts","Interés en conciertos","Ordinal","1–10","Time 1"),
    ("music","Interés en música","Ordinal","1–10","Time 1"),
    ("shopping","Interés en compras","Ordinal","1–10","Time 1"),
    ("yoga","Interés en yoga/meditación","Ordinal","1–10","Time 1"),
    ("match","TARGET: Match mutuo","Binaria","0=No, 1=Sí","Scorecard"),
], columns=["Variable","Descripción","Tipo","Valores válidos","Momento captura"])

print(f"Total variables seleccionadas: {len(diccionario)}")
diccionario

Total variables seleccionadas: 47


,Variable,Descripción,Tipo,Valores válidos,Momento captura
0,gender,Género del participante,Binaria,"0=Mujer, 1=Hombre",Time 1
1,age,Edad del participante,Numérica,18–45,Time 1
2,age_o,Edad de la pareja,Numérica,18–45,Time 1
3,race,Raza del participante,Categórica,"1=Negro,2=Caucásico,3=Latino,4=Asiático,6=Otro",Time 1
4,race_o,Raza de la pareja,Categórica,1-6,Time 1
5,samerace,Misma raza que la pareja,Binaria,"0=No, 1=Sí",Time 1
6,imprace,Importancia de misma raza en pareja ideal,Ordinal,1–10,Time 1
7,imprelig,Importancia de misma religión en pareja,Ordinal,1–10,Time 1
8,goal,Objetivo en el evento,Categórica,1=Diversión…6=Otro,Time 1
9,date,Frecuencia de citas en general,Ordinal,1=Varias/sem…7=Nunca,Time 1
